# Hands-on 2 — From a density field to the cosmic web

Displace H1's field, watch a web appear, classify it, and find out how much of
it Zel'dovich had no business describing.

| # | step | ~min |
|---|---|---|
| 1 | rebuild the field from CAMB | 10 |
| 2 | the Zel'dovich displacement | 12 |
| 3 | project a slab — the web appears | 15 |
| 4 | classify it, and check the approximation | 16 |
| 5 | colour the slab | 10 |
| 6 | second order: 2LPT | 15 |
| 7 | benchmark against FlowPM | 10 |

Cells marked `# TODO` are yours. The cell below each one holds the answer and
is collapsed — expand it if you want it. Everything runs either way.

#### Cosmology, grid, and the linear spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Om, Ob, h, ns, sigma8, Tcmb = 0.31, 0.048, 0.676, 0.965, 0.81, 2.7255
N, L, SEED = 128, 250.0, 1234
k_f, k_Nyq = 2*np.pi/L, np.pi*N/L

try:
    import camb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "camb"], check=False)
    try:
        import camb
    except ImportError:
        camb = None


def pk_from_camb():
    pars = camb.CAMBparams()
    pars.set_cosmology(H0=100*h, ombh2=Ob*h*h, omch2=(Om - Ob)*h*h,
                       mnu=0.0, omk=0, num_massive_neutrinos=0, TCMB=Tcmb)
    pars.InitPower.set_params(ns=ns, As=2.1e-9)
    pars.set_matter_power(redshifts=[0.0], kmax=60.0)
    pars.NonLinear = camb.model.NonLinear_none
    kk, _, pp = camb.get_results(pars).get_matter_power_spectrum(
        minkh=1e-4, maxkh=50.0, npoints=1024)
    return kk, pp[0]


def sigma_R(k, P, R=8.0):
    x = k*R
    W = 3*(np.sin(x) - x*np.cos(x))/x**3
    return np.sqrt(np.trapz(k**3*P*W**2/(2*np.pi**2), np.log(k)))


if camb is not None:
    ktab, ptab = pk_from_camb()
    source = "computed with CAMB"
else:
    import urllib.request
    url = ("https://raw.githubusercontent.com/MinhMPA/EFT-with-FFT/"
           "master/notebooks/pk_lin_fiducial.txt")
    try:
        tab = np.loadtxt("pk_lin_fiducial.txt")
    except OSError:
        urllib.request.urlretrieve(url, "pk_lin_fiducial.txt")
        tab = np.loadtxt("pk_lin_fiducial.txt")
    ktab, ptab = tab[:, 0], tab[:, 1]
    source = "from the shipped CAMB table"

ptab = ptab*(sigma8/sigma_R(ktab, ptab))**2      # normalise to sigma_8 = 0.81
pk_lin = lambda q: np.exp(np.interp(np.log(q), np.log(ktab), np.log(ptab)))
print(f"P_L(k) {source};  sigma_8 = {sigma_R(ktab, ptab):.4f}")

#### The same field H1 drew

Same seed, same box, same recipe — so this is H1's field, rebuilt rather than
loaded. Nothing you did in H1 needs to have worked.

In [ ]:
kx = np.fft.fftfreq(N, d=1.0/N)*k_f
kz = np.fft.rfftfreq(N, d=1.0/N)*k_f
KX, KY, KZ = np.meshgrid(kx, kx, kz, indexing="ij")
K2 = KX**2 + KY**2 + KZ**2
K2[0, 0, 0] = 1.0

P_grid = pk_lin(np.sqrt(K2).ravel()).reshape(K2.shape)
P_grid[0, 0, 0] = 0.0

rng     = np.random.default_rng(SEED)
delta_k = np.fft.rfftn(rng.standard_normal((N, N, N)))*np.sqrt(P_grid*N**3/L**3)
delta_k[0, 0, 0] = 0.0
delta_x = np.fft.irfftn(delta_k, s=(N, N, N))

print(f"rms delta = {np.std(delta_x):.4f}")     # quiz: H1 got 2.516 with Eisenstein & Hu.
                                                #       Why is this one different?
assert 2.50 < np.std(delta_x) < 2.56, f"expected ~2.53, got {np.std(delta_x):.4f}"

#### Growth factors

`Ψ⁽¹⁾ ∝ D₁` and `Ψ⁽²⁾ ∝ D₁²`, so one set of FFTs serves both epochs.

In [ ]:
from scipy.integrate import quad          # only for the growth integral


def D1(z):
    """Linear growth, normalised to D1(0) = 1."""
    def integrand(a):
        return 1.0/(a*np.sqrt(Om/a**3 + (1 - Om)))**3
    def D(a):
        E = np.sqrt(Om/a**3 + (1 - Om))
        return 2.5*Om*E*quad(integrand, 1e-8, a, limit=200)[0]
    return D(1.0/(1 + z))/D(1.0)


for z in (49, 9, 1, 0):
    print(f"  z = {z:3d}   D1 = {D1(z):.4f}")

## Step 2 — The Zel'dovich displacement

Notes §2.2: every particle starts on a grid point $\boldsymbol{q}$ and moves to

$$\boldsymbol{x} = \boldsymbol{q} + D_1(z)\,\boldsymbol{\Psi}^{(1)}(\boldsymbol{q}),
\qquad \boldsymbol{\Psi}^{(1)}(\boldsymbol{k}) = \frac{i\boldsymbol{k}}{k^2}\,\delta(\boldsymbol{k}).$$

That is the unique curl-free field with $\nabla\cdot\boldsymbol{\Psi}^{(1)} = -\delta$,
which is what the checkpoint tests.

In [ ]:
# TODO: Psi^(1)(k) = i k / k^2 * delta(k), for each of the three axes.
# Watch the sign: getting it backwards makes matter flow OUT of overdensities,
# and the rms will not tell you.
psi1_k = [...,  ...,  ...]

if not any(p is ... for p in psi1_k):   # skips quietly until you fill the dots
    psi1 = [np.fft.irfftn(p, s=(N, N, N)) for p in psi1_k]

In [ ]:
#@title Solution — Psi^(1)
psi1_k = [1j*Ki/K2*delta_k for Ki in (KX, KY, KZ)]
psi1   = [np.fft.irfftn(p, s=(N, N, N)) for p in psi1_k]

In [ ]:
# --- checkpoint: div Psi = -delta ---------------------------------------
F = np.fft.rfftn
div = np.fft.irfftn(1j*(KX*F(psi1[0]) + KY*F(psi1[1]) + KZ*F(psi1[2])), s=(N, N, N))
err = np.abs(div + delta_x).max()/np.abs(delta_x).max()

print(f"max |div.Psi + delta| / max|delta| = {err:.4f}")
print(f"rms Psi per axis at z=0            = "
      f"{[round(float(np.std(p)), 3) for p in psi1]}")

assert err < 0.05, f"div.Psi should equal -delta; got {err:.3f}"

The residual is **0.028**, not machine zero, and that is the **Nyquist plane**:
for even $N$ the mode at $k_{\rm Nyq}$ appears once with an ambiguous sign, so
$ik$ is not exactly antisymmetric there.

Note the three axes give **5.203, 4.828, 6.091** — a ±12% spread on identical
correct code, because $\Psi$ is dominated by the few longest modes the box holds.

#### What a finite box costs you

In [ ]:
kk = np.logspace(-5, np.log10(50), 4000)
allk = np.sqrt(np.trapz(pk_lin(kk), kk)/(6*np.pi**2))
kb   = np.logspace(np.log10(k_f), np.log10(k_Nyq), 4000)
box  = np.sqrt(np.trapz(pk_lin(kb), kb)/(6*np.pi**2))
below = np.trapz(pk_lin(kk[kk < k_f]), kk[kk < k_f])/np.trapz(pk_lin(kk), kk)
below2 = (np.trapz(kk[kk < k_f]**2*pk_lin(kk[kk < k_f]), kk[kk < k_f])
          / np.trapz(kk**2*pk_lin(kk), kk))

print(f"rms Psi_x   realized            {np.std(psi1[0]):.3f} Mpc/h")
print(f"            continuum, all k    {allk:.3f}   -> you are {100*(1-np.std(psi1[0])/allk):.0f}% LOW")
print(f"            continuum, k_f..kNy {box:.3f}   -> you are {100*(np.std(psi1[0])/box-1):.0f}% HIGH")
print(f"\nfraction of int dk P     below k_f: {100*below:5.1f}%")
print(f"fraction of int dk k^2 P below k_f: {100*below2:5.2f}%")
# quiz: Psi = delta/k, so the displacement integral has no k-weighting and the
#       density integral has k^2. Which of the two numbers above explains the 11%?

$\langle\Psi_x^2\rangle = \frac{1}{6\pi^2}\int{\rm d}k\,P(k)$ has **no
$k$-weighting**; $\langle\delta^2\rangle$ carries $k^2$. So the box throws away
**24%** of what makes displacements and **0.01%** of what makes density
contrast. Displacement is the quantity that notices a finite box.

## Step 3 — Project a slab

Every particle starts on the grid at $\boldsymbol{q}$ and moves to
$\boldsymbol{x} = \boldsymbol{q} + D_1(z)\,\boldsymbol{\Psi}^{(1)}(\boldsymbol{q})$.
Below, that displacement is applied at two epochs — $z=49$, close to the
initial conditions, and $z=0$, today — and a thin slab of each is projected
onto the plane.

In [ ]:
q = (np.arange(N) + 0.5)*(L/N)
Q = np.meshgrid(q, q, q, indexing="ij")

pos49 = [(Q[i] + D1(49)*psi1[i]).ravel() % L for i in range(3)]
pos0  = [(Q[i] + D1(0) *psi1[i]).ravel() % L for i in range(3)]

In [ ]:
TH = 15.0     # h^-1 Mpc slab thickness
fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
Hs = []
for pos in (pos49, pos0):
    sel = pos[2] < TH
    Hc, _, _ = np.histogram2d(pos[0][sel], pos[1][sel], bins=400, range=[[0, L], [0, L]])
    Hs.append(np.log10(Hc.T + 1))
vmin, vmax = min(H.min() for H in Hs), max(H.max() for H in Hs)
ims = []
for a, Hlog, z in zip(ax, Hs, (49, 0)):
    im = a.imshow(Hlog, origin="lower", extent=[0, L, 0, L], cmap="bone_r",
                  interpolation="nearest", vmin=vmin, vmax=vmax)
    ims.append(im)
    a.set_title(f"z = {z}")
    a.set_xlabel(r"$x\ [h^{-1}\,{\rm Mpc}]$")
    a.set_ylabel(r"$y\ [h^{-1}\,{\rm Mpc}]$")
fig.suptitle(f"{TH:.0f} h$^{{-1}}$ Mpc slab")
fig.colorbar(ims[-1], ax=ax, label="log10(N + 1)", shrink=0.85)
plt.show()

# quiz: the left panel looks like a faint grid, the right like a web.
#       rms displacement is 0.07 cells at z=49 and 2.8 cells at z=0.
#       What sets the scale of the pattern you see in each panel?

At $z=49$ the displacement is about a fifteenth of a cell, so the picture is
just the Lagrangian grid, barely perturbed. At $z=0$ the web has appeared —
sheets, filaments, knots — and step 4 asks what kind of place each particle
landed in.

## Step 4 — What kind of place is each particle in?

Notes §2.3. The deformation tensor is
$D_{ij}(\boldsymbol{k}) = k_ik_j\,\delta(\boldsymbol{k})/k^2$, and the sign of
its three eigenvalues says whether a fluid element is collapsing along that
axis. Count the collapsing axes and you get void, sheet, filament, knot.

**Build it from the field you started with, not the displaced one.**
Doroshkevich's `8/42/42/8` is a theorem about *Gaussian* fields, and displacing
destroys Gaussianity. Classify first, move second — each particle carries its
label to wherever $\Psi$ puts it.

In [ ]:
ks = (KX, KY, KZ)
Mt = np.empty((N**3, 3, 3), dtype=np.float32)
for i in range(3):
    for j in range(i, 3):
        Mt[:, i, j] = Mt[:, j, i] = np.fft.irfftn(
            ks[i]*ks[j]/K2*delta_k, s=(N, N, N)).ravel()
lam = np.linalg.eigvalsh(Mt)          # ascending: lam[:,0] <= lam[:,1] <= lam[:,2]
del Mt
print(f"eigenvalues: {lam.shape},  lam_max over the box = {lam[:,2].max():.2f}")

In [ ]:
# TODO: how many of the three eigenvalues are positive, per particle?
# 0 -> void, 1 -> sheet, 2 -> filament, 3 -> knot.
npos = ...

In [ ]:
#@title Solution — count the collapsing axes
npos = (lam > 0).sum(axis=1)

In [ ]:
# --- checkpoint: Doroshkevich ------------------------------------------
frac = [100*(npos == n).mean() for n in range(4)]
for n, lab in enumerate(["void", "sheet", "filament", "knot"]):
    print(f"  {n} collapsing axes   {lab:9s} {frac[n]:5.1f}%")
print("  Doroshkevich (1970):            8.0  42.0  42.0   8.0")

for n, want in enumerate((8.0, 42.0, 42.0, 8.0)):
    assert abs(frac[n] - want) < 1.5, f"{n}: expected {want}, got {frac[n]:.1f}"
# quiz: these fractions test Gaussianity and isotropy, almost nothing else.
#       A sign error gives the SAME numbers with every label silently
#       swapped -- 8/42/42/8 is a palindrome -- and dropping k^2 barely
#       moves them either: the fractions ignore the radial weight. Step 5's
#       coloured picture is the real test: knots must sit on the nodes.
#       What would classifying the *displaced* field give?

#### Was any of this legitimate?

Zel'dovich assumes streams never cross. The Jacobian of
$\boldsymbol{q}\mapsto\boldsymbol{x}$ is $\prod_i(1 - D_1\lambda_i)$, so a
particle has shell-crossed once $D_1\lambda_{\max} > 1$.

In [ ]:
print("   z     D1      shell-crossed")
for z in (49, 9, 1, 0):
    d = D1(z)
    print(f"  {z:3d}  {d:.4f}      {100*(d*lam[:, 2] > 1).mean():6.2f}%")
Dc = 1.0/lam[:, 2].max()
print(f"\nfirst crossing anywhere in the box at D1 = {Dc:.4f}")
# quiz: two thirds of the box has shell-crossed by z=0. Why plot it anyway?
#       And what does an N-body code do that Zel'dovich cannot?

**63.9% by z = 0**, and the first crossing at $D_1 = 0.163$, about z ≈ 6.8. The
picture in step 3 is one where the approximation has failed across most of the
volume — which is §2.4 of the notes, arriving as a measurement rather than a
claim. It is also why 2LPT will sharpen the filaments and then overshoot.

## Step 5 — Colour the slab

Each particle was classified back at its Lagrangian position $\boldsymbol{q}$
(step 4) and then displaced to $z=0$ (step 3). Below is the same slab as
before, but now every particle is drawn in the colour of its class.

In [ ]:
sel = pos0[2] < 15.0
sub = np.random.default_rng(1).permutation(np.flatnonzero(sel))[:400000]
cls = npos.ravel()[sub]

cols = ["#d9d9d9", "#7fbf8f", "#2f6ea5", "#e8590c"]     # void, sheet, filament, knot
labs = ["void", "sheet", "filament", "knot"]

fig, ax = plt.subplots(figsize=(6, 6))
for n, (cl, sz) in enumerate(zip(cols, [0.20, 0.28, 0.35, 0.55])):
    m = cls == n
    ax.scatter(pos0[0][sub][m], pos0[1][sub][m], c=cl, s=sz, marker=".",
               linewidths=0, rasterized=True, zorder=2 + n)

ax.set_xlim(0, L)
ax.set_ylim(0, L)
ax.set_aspect("equal")
ax.set_title("Cosmic web at z = 0, coloured by class")
ax.set_xlabel(r"$x\ [h^{-1}\,{\rm Mpc}]$")
ax.set_ylabel(r"$y\ [h^{-1}\,{\rm Mpc}]$")

handles = [plt.Line2D([0], [0], marker="s", linestyle="", color=cols[n],
                       label=f"{labs[n]} ({frac[n]:.0f}%)") for n in range(4)]
ax.legend(handles=handles, loc="upper right", fontsize=8, facecolor="white")
plt.tight_layout()
plt.show()

# quiz: the knots sit at the nodes where filaments meet, and the filaments
#       connect them. Nothing in step 4 knew about positions -- the classes
#       came from the INITIAL field. Why does the geometry come out right anyway?

Nothing in step 4 knew about positions — the classes were assigned before
anything moved. Yet the knots land on the nodes and the filaments trace the
strands, because the tidal field that classifies a region is the same field
that later collapses it: $\Psi$ and $D_{ij}$ come from the same $\delta(k)$.
This is the Lecture 2 figure, built here from your own code, and step 6
sharpens it further.

## Step 6 — Second order

Notes §2.5. The second-order displacement is sourced by

$$\delta^{(2)} = \sum_{i<j}\left[\varphi_{,ii}\varphi_{,jj} - \varphi_{,ij}^2\right],
\qquad \nabla^2\varphi = \delta,$$

and then $\boldsymbol{\Psi}^{(2)} = \tfrac{3}{7}D_1^2\,\nabla\nabla^{-2}\delta^{(2)}$.
The $3/7$ is the Einstein–de Sitter value of the second-order growth ratio.

In [ ]:
phi_k = -delta_k/K2                      # lap phi = delta
phi = {}
for i in range(3):
    for j in range(i, 3):
        phi[(i, j)] = np.fft.irfftn(-ks[i]*ks[j]*phi_k, s=(N, N, N))

In [ ]:
# TODO: delta2 = sum over i<j of [ phi_,ii * phi_,jj  -  phi_,ij^2 ]
# phi[(i,j)] holds phi_,ij for i <= j.
delta2 = np.zeros((N, N, N))
...

In [ ]:
#@title Solution — the second-order source
delta2 = np.zeros((N, N, N))
for i in range(3):
    for j in range(i + 1, 3):
        delta2 += phi[(i, i)]*phi[(j, j)] - phi[(i, j)]**2

In [ ]:
delta2_k = np.fft.rfftn(delta2)
psi2 = [np.fft.irfftn(-1j*ks[i]/K2*delta2_k, s=(N, N, N)) for i in range(3)]

r1 = np.sqrt(sum(np.var(p) for p in psi1)/3)
r2 = np.sqrt(sum(np.var(p) for p in psi2)/3)
print("   z     |Psi1|    (3/7)|Psi2|   ratio")
for z in (49, 0):
    d = D1(z)
    print(f"  {z:3d}   {d*r1:7.3f}   {3/7*d**2*r2:9.3f}   {3/7*d**2*r2/(d*r1):6.3f}")

assert abs(3/7*r2/r1 - 0.181) < 0.02, "2LPT/1LPT at z=0 should be ~0.18"
# quiz: the ratio is 0.5% at z=49 and 18% at z=0. Which epoch is LPT for?

The correction is **0.5%** of $\Psi^{(1)}$ at $z=49$ and **18%** at $z=0$.
2LPT is a perturbative correction to Zel'dovich, and a perturbative correction
is trustworthy exactly where it stays small. That is the early-time regime the
initial conditions sit in, not today. By $z=0$ the "correction" is no longer
small, and that is the same warning step 4 already gave from a different
angle: **63.9%** of the box has shell-crossed by $z=0$, so both numbers are
pointing at the same breakdown of the perturbative expansion.

In [ ]:
pos2 = [(pos0[i] + 3/7*D1(0)**2*psi2[i].ravel()) % L for i in range(3)]

fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
Hs = []
for pos in (pos0, pos2):
    sel = pos[2] < 15.0
    Hc, _, _ = np.histogram2d(pos[0][sel], pos[1][sel], bins=400, range=[[0, L], [0, L]])
    Hs.append(np.log10(Hc.T + 1))
vmin, vmax = min(H.min() for H in Hs), max(H.max() for H in Hs)
ims = []
for a, Hlog, title in zip(ax, Hs, ("Zel'dovich", "Zel'dovich + 2LPT")):
    im = a.imshow(Hlog, origin="lower", extent=[0, L, 0, L], cmap="bone_r",
              interpolation="nearest", vmin=vmin, vmax=vmax)
    ims.append(im)
    a.set_title(title)
    a.set_xlabel(r"$x\ [h^{-1}\,{\rm Mpc}]$")
    a.set_ylabel(r"$y\ [h^{-1}\,{\rm Mpc}]$")
fig.suptitle("15 h$^{-1}$ Mpc slab")
fig.colorbar(ims[-1], ax=ax, label="log10(N + 1)", shrink=0.85)
plt.show()

# quiz: are the filaments and knots on the right sharper than on the left?
#       Given that 63.9% of the box has already shell-crossed by z=0 (step 4),
#       is "sharper" here trustworthy, or could 2LPT be oversharpening a
#       region where the perturbative expansion has already broken down?

Yes, the right panel's filaments and knots are visibly thinner and denser than
the left panel's, because 2LPT sharpens the same single-stream skeleton
Zel'dovich already traced. That sharpening is not automatically trustworthy.
2LPT is still a single-stream, perturbative displacement, with no mechanism
for multi-streaming, and **63.9%** of the box has already shell-crossed by
$z=0$ (step 4). In that two-thirds of the volume, "sharper" means 2LPT is
pushing particles further along a trajectory a real particle already left —
the overshoot step 4 anticipated, not a more correct picture.

## Step 7 — Benchmark against FlowPM

[FlowPM](https://github.com/DifferentiableUniverseInitiative/flowpm) is a
TensorFlow N-body code that implements the same 2LPT you just coded by hand.
`flowpm.tfpm.lpt2_source` computes exactly

$$\text{source} = \frac{3}{7}\sum_{i<j}\left[\varphi_{,ii}\varphi_{,jj} - \varphi_{,ij}^2\right],$$

your `delta2` with the $3/7$ already folded in — the same quantity, not a
downstream displacement, so the comparison below is direct.

**One convention has to be matched, not compared.** FlowPM's gradient kernel
is a hardcoded finite difference, $a(w) = \tfrac{1}{6}(8\sin w - \sin 2w)$
with $w = kL/N$ in grid units, and `lpt2_source` gives no option to switch it
to spectral. Its Laplacian, by contrast, is the exact $1/\sum_i w_i^2$ — the
same formula you used. So the cell below recomputes `delta2` a second time
using FlowPM's own finite-difference gradient throughout, Laplacian included,
entirely in grid units: an earlier attempt that kept the Laplacian in
physical $k$ and swapped only the gradient was off by a clean factor of
$(L/N)^4$, which is the kind of thing an experiment catches and eyeballing
does not. With everything on the same footing, what is compared is the
**2LPT algebra alone** — the gradient convention is held fixed between the
two sides.

**Be honest about what this validates.** Your `delta2` above used the
spectral gradient the lecture derived; that is not the array being checked
here. What is checked is whether the second-order source you assembled —
$\varphi_{,ii}\varphi_{,jj} - \varphi_{,ij}^2$, summed over $i<j$ — is the
same algebra an independently developed, production N-body code encodes.
"Your code matches FlowPM" would overclaim; "the algorithm matches FlowPM's,
under FlowPM's own gradient" is what is actually shown.

In [ ]:
try:
    import flowpm
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "tensorflow", "tensorflow_probability", "tf-keras", "flowpm"],
                   check=False)
    try:
        import flowpm
    except ImportError:
        flowpm = None

if flowpm is not None:
    import tensorflow as tf
    print(f"flowpm ready (tensorflow {tf.__version__})")
else:
    print("flowpm unavailable -- the benchmark cell below will skip")

In [ ]:
if flowpm is None:
    print("flowpm not installed -- skipping the FlowPM benchmark.")
else:
    from flowpm.utils import r2c3d, c2r3d
    from flowpm.tfpm import lpt2_source

    dlin_k = r2c3d(tf.convert_to_tensor(delta_x[None].astype(np.float32)), norm=N**3)
    theirs = c2r3d(lpt2_source(dlin_k), norm=N**3).numpy()[0]

    # FlowPM's gradient: a(w) = (8 sin w - sin 2w)/6, w = k*L/N (grid units).
    # Its Laplacian is the exact 1/sum(w_i^2) -- same formula as ours, but
    # evaluated on the same grid-unit w so both kernels are self-consistent.
    wgrid = [ki*L/N for ki in ks]
    agrid = [(8*np.sin(wi) - np.sin(2*wi))/6 for wi in wgrid]
    W2 = sum(wi**2 for wi in wgrid)
    W2[0, 0, 0] = 1.0
    lapgrid = 1.0/W2
    lapgrid[0, 0, 0] = 0.0

    phiFD = {}
    for i in range(3):
        for j in range(i, 3):
            phiFD[(i, j)] = np.fft.irfftn(-lapgrid*agrid[i]*agrid[j]*delta_k, s=(N, N, N))
    delta2_FD = np.zeros((N, N, N))
    for i in range(3):
        for j in range(i + 1, 3):
            delta2_FD += phiFD[(i, i)]*phiFD[(j, j)] - phiFD[(i, j)]**2
    ours = 3/7*delta2_FD

    denom = np.abs(theirs).max()
    max_rel = np.abs(ours - theirs).max()/denom
    honesty = np.sqrt(np.mean((3/7*delta2 - theirs)**2))/denom

    print(f"FD-matched agreement:  max|ours - FlowPM| / max|FlowPM| = {max_rel:.2e}")
    print(f"spectral (yours) vs FlowPM's FD convention: rms diff / max|FlowPM| = {honesty:.3f}")
    print("  -- the price of a finite grid's gradient convention, not a bug; cf. step 2's Nyquist residual")

    assert max_rel < 1e-4, f"FD-matched delta_2 should match FlowPM to ~1e-4; got {max_rel:.2e}"

You built this pipeline yourself, start to finish: a linear power spectrum
became a field, the field became a displacement, the displacement became the
cosmic web, the web was classified and coloured by the same tidal field that
made it — and now the one piece of algebra sitting underneath all of it has
been checked against a production N-body code.